# 59 — LGBM Extreme Cliff Oversampling + SMILES Augmentation

Extreme cliff-focused oversampling: 20x for actives (pEC50 >= 6), 15x for cliff pairs,
plus SMILES augmentation (randomized atom ordering via RDKit) for active compounds.

Key constraint: augmented copies of a compound go to the **same fold** as the original
to prevent leakage — assignment is by original compound InChIKey.

Key steps:
1. Load data + cliff labels.
2. SMILES augmentation function (randomized canonical SMILES).
3. Extreme oversampling with fold-safe augmentation.
4. Scaffold 5-fold CV.
5. Save OOF + submission.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdmolfiles

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, to_inchikey
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')

Train: 4,139  Test: 513


## 1. Load data + cliff labels

In [2]:
# ── Load cliff labels ─────────────────────────────────────────────────────────
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
if cliff_path.exists():
    cliff_df = pd.read_parquet(cliff_path)
    # Merge by name or smiles
    if 'name' in cliff_df.columns and 'name' in tr.columns:
        tr = tr.merge(cliff_df[['name', 'cliff_role']].drop_duplicates('name'),
                      on='name', how='left')
    elif 'smiles' in cliff_df.columns:
        tr = tr.merge(cliff_df[['smiles', 'cliff_role']].drop_duplicates('smiles'),
                      on='smiles', how='left')
    tr['cliff_role'] = tr.get('cliff_role', pd.Series(dtype=int)).fillna(0).astype(int)
    print(f'Loaded cliff labels: {(tr["cliff_role"] != 0).sum()} cliff members')
else:
    print('cliff_labels.parquet not found — computing inline...')
    from pxr.chem import morgan_fp_batch
    fp_mat = morgan_fp_batch(tr['smiles'].tolist()).astype(np.float32)
    y_vals = tr['pec50'].values
    cliff_role = np.zeros(len(tr), dtype=np.int8)
    n = len(tr)
    BATCH = 256
    for i in range(0, n, BATCH):
        chunk = fp_mat[i:i+BATCH]
        dot = chunk @ fp_mat.T
        union = chunk.sum(1, keepdims=True) + fp_mat.sum(1)[None, :] - dot
        with np.errstate(divide='ignore', invalid='ignore'):
            tan = np.where(union > 0, dot / union, 0.0)
        for bi, gi in enumerate(range(i, min(i+BATCH, n))):
            for j in range(n):
                if gi == j:
                    continue
                if tan[bi, j] >= 0.6 and abs(y_vals[gi] - y_vals[j]) >= 1.0:
                    cliff_role[gi] = 1 if y_vals[gi] > y_vals[j] else -1
    tr['cliff_role'] = cliff_role
    cliff_df_out = tr[['smiles', 'cliff_role']].copy()
    cliff_df_out.to_parquet(DATA_PROCESSED / 'cliff_labels.parquet', index=False)
    print(f'Computed and saved cliff labels: {(cliff_role != 0).sum()} cliff members')

# ── Basic statistics ──────────────────────────────────────────────────────────
y_tr = tr['pec50'].values.astype(np.float32)
is_active = y_tr >= 6.0
is_cliff  = tr['cliff_role'].values != 0
n_active = is_active.sum()
n_cliff  = is_cliff.sum()
print(f'\nActive compounds (pEC50 >= 6): {n_active} ({n_active/len(tr)*100:.1f}%)')
print(f'Cliff members:                  {n_cliff} ({n_cliff/len(tr)*100:.1f}%)')

Loaded cliff labels: 248 cliff members

Active compounds (pEC50 >= 6): 67 (1.6%)
Cliff members:                  248 (6.0%)


## 2. Extreme oversampling + SMILES augmentation

In [3]:
def augment_smiles(smi: str, n: int = 5, seed: int = 42) -> list[str]:
    """Generate up to n non-canonical SMILES by randomized atom ordering."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return []
    results = set()
    n_atoms = mol.GetNumAtoms()
    for i in range(n * 3):
        root_atom = (i + seed) % max(n_atoms, 1)
        try:
            aug = rdmolfiles.MolToSmiles(mol, doRandom=True, rootedAtAtom=root_atom)
            if aug and aug != smi:
                results.add(aug)
        except Exception:
            pass
        if len(results) >= n:
            break
    return list(results)[:n]


# ── Build augmented training records ─────────────────────────────────────────
# Strategy:
#   Active non-cliff (pEC50 >= 6): 20x + up to 5 SMILES augmentations (w=0.8)
#   Cliff members: 15x (both active and inactive cliff sides)
#   All others: 1x (no duplication)

OVERSAMPLE_ACTIVE = 20
OVERSAMPLE_CLIFF  = 15
AUG_N             = 5
AUG_WEIGHT        = 0.8

aug_smiles_list   = []
aug_pec50_list    = []
aug_weight_list   = []
aug_inchikey_list = []  # track original compound for fold assignment

inchikeys = tr['smiles'].map(to_inchikey).tolist()

for i in range(len(tr)):
    smi = tr.iloc[i]['smiles']
    pec = float(y_tr[i])
    cliff_role = int(tr.iloc[i].get('cliff_role', 0))
    ik = inchikeys[i] or f'__missing_{i}__'

    # Determine oversampling multiplier
    if cliff_role != 0:
        n_copies = OVERSAMPLE_CLIFF
        base_w = 1.0
    elif pec >= 6.0:
        n_copies = OVERSAMPLE_ACTIVE
        base_w = 1.0
    else:
        n_copies = 1
        base_w = 1.0

    # Original + copies
    for _ in range(n_copies):
        aug_smiles_list.append(smi)
        aug_pec50_list.append(pec)
        aug_weight_list.append(base_w)
        aug_inchikey_list.append(ik)

    # SMILES augmentation for actives
    if pec >= 6.0:
        augmented = augment_smiles(smi, n=AUG_N, seed=SEED + i)
        for aug_smi in augmented:
            aug_smiles_list.append(aug_smi)
            aug_pec50_list.append(pec)
            aug_weight_list.append(AUG_WEIGHT)
            aug_inchikey_list.append(ik)  # same original compound = same fold

print(f'Augmented training set: {len(aug_smiles_list):,} records')
print(f'  Original (1x):     {(np.array(aug_weight_list) == 1.0).sum():,}')
print(f'  SMILES augmented:  {(np.array(aug_weight_list) == AUG_WEIGHT).sum():,}')
print(f'Original training set size: {len(tr):,}')

Augmented training set: 9,143 records
  Original (1x):     8,808
  SMILES augmented:  335
Original training set size: 4,139


## 3. Fold-safe featurization

In [4]:
print('Featurizing augmented training set (this may take a few minutes)...')
X_aug_all = impute(combined(aug_smiles_list))
y_aug_all  = np.array(aug_pec50_list, dtype=np.float32)
w_aug_all  = np.array(aug_weight_list, dtype=np.float32)
ik_aug_all = np.array(aug_inchikey_list)
print(f'X_aug_all: {X_aug_all.shape}')

# Featurize original training set (for OOF evaluation)
print('Featurizing original training set...')
X_tr_orig = impute(combined(tr['smiles'].tolist()))
print(f'X_tr_orig: {X_tr_orig.shape}')

# Featurize test set
print('Featurizing test set...')
X_te = impute(combined(te['smiles'].tolist()))
print(f'X_te: {X_te.shape}')

# Build scaffold splits on ORIGINAL training set
scaffolds_orig = tr['smiles'].map(bemis_murcko).tolist()
splits_orig = scaffold_kfold_indices(scaffolds_orig, n_splits=N_FOLDS, seed=SEED)

# Build InChIKey-to-fold mapping from original splits
orig_inchikeys = np.array([inchikeys[i] or f'__missing_{i}__' for i in range(len(tr))])
ik_to_fold = {}
for fold_idx, (tr_idx, va_idx) in enumerate(splits_orig):
    for vi in va_idx:
        ik_to_fold[orig_inchikeys[vi]] = fold_idx

# Assign each augmented sample to its original compound's fold
aug_fold_assignments = np.array([ik_to_fold.get(ik, 0) for ik in ik_aug_all])
print(f'\nFold size distribution (augmented):')
for f in range(N_FOLDS):
    n_f = (aug_fold_assignments == f).sum()
    print(f'  Fold {f}: {n_f:,} samples in validation slot')

Featurizing augmented training set (this may take a few minutes)...


X_aug_all: (9143, 2265)
Featurizing original training set...


X_tr_orig: (4139, 2265)
Featurizing test set...


X_te: (513, 2265)



Fold size distribution (augmented):
  Fold 0: 1,939 samples in validation slot
  Fold 1: 1,911 samples in validation slot
  Fold 2: 1,642 samples in validation slot
  Fold 3: 1,734 samples in validation slot
  Fold 4: 1,917 samples in validation slot


## 4. Scaffold 5-fold CV

In [5]:
# ── OOF on ORIGINAL compounds (for fair comparison) ───────────────────────────
# Training fold: all augmented samples NOT in that fold's validation InChIKeys
# Validation: original compounds in the validation fold (evaluate on original SMILES)

oof_oversample = np.full(len(tr), np.nan, dtype=np.float32)
oof_baseline   = np.full(len(tr), np.nan, dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(splits_orig):
    # Augmented training set: all rows NOT assigned to this fold
    train_mask = aug_fold_assignments != fold
    X_fold_tr = X_aug_all[train_mask]
    y_fold_tr = y_aug_all[train_mask]
    w_fold_tr = w_aug_all[train_mask]

    # Validation: original (non-augmented) compounds in the val fold
    X_fold_va = X_tr_orig[va_idx]
    y_fold_va = y_tr[va_idx]

    # Oversampled model
    m_over = lgb.LGBMRegressor(**LGBM_PARAMS)
    m_over.fit(X_fold_tr, y_fold_tr, sample_weight=w_fold_tr,
               callbacks=[lgb.log_evaluation(-1)])
    oof_oversample[va_idx] = m_over.predict(X_fold_va)

    # Baseline (no augmentation, uniform weights)
    m_base = lgb.LGBMRegressor(**LGBM_PARAMS)
    m_base.fit(X_tr_orig[tr_idx], y_tr[tr_idx],
               callbacks=[lgb.log_evaluation(-1)])
    oof_baseline[va_idx] = m_base.predict(X_fold_va)

    rae_over = rae(y_fold_va, oof_oversample[va_idx])
    rae_base = rae(y_fold_va, oof_baseline[va_idx])
    print(f'  Fold {fold+1}: baseline RAE={rae_base:.4f}  oversampled RAE={rae_over:.4f}')

rae_over_oof = rae(y_tr, oof_oversample)
rae_base_oof = rae(y_tr, oof_baseline)

# Also compute cliff-specific RAE
if is_cliff.sum() > 5:
    rae_cliff_over = rae(y_tr[is_cliff], oof_oversample[is_cliff])
    rae_cliff_base = rae(y_tr[is_cliff], oof_baseline[is_cliff])
    print(f'\nCliff-member OOF RAE: baseline={rae_cliff_base:.4f}  oversampled={rae_cliff_over:.4f}')

print(f'\nOverall OOF RAE: baseline={rae_base_oof:.4f}  oversampled={rae_over_oof:.4f}')
print(f'Delta:           {rae_over_oof - rae_base_oof:+.4f}')

  Fold 1: baseline RAE=0.4934  oversampled RAE=0.4957


  Fold 2: baseline RAE=0.5762  oversampled RAE=0.6079


  Fold 3: baseline RAE=0.5961  oversampled RAE=0.6131


  Fold 4: baseline RAE=0.5635  oversampled RAE=0.5725


  Fold 5: baseline RAE=0.5952  oversampled RAE=0.6156

Cliff-member OOF RAE: baseline=0.6163  oversampled=0.6237

Overall OOF RAE: baseline=0.5600  oversampled=0.5757
Delta:           +0.0157


## 5. Save

In [6]:
# Final model: train on all augmented data
print(f'Training final model on {len(X_aug_all):,} augmented samples...')
final_model = lgb.LGBMRegressor(**LGBM_PARAMS)
final_model.fit(X_aug_all, y_aug_all, sample_weight=w_aug_all,
                callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(final_model.predict(X_te),
                   float(y_tr.min()) - 0.5, float(y_tr.max()) + 0.5)

best_oof = oof_oversample if rae_over_oof <= rae_base_oof else oof_baseline
best_label = 'oversampled' if rae_over_oof <= rae_base_oof else 'baseline'

np.save(DATA_PROCESSED / 'oof_lgbm_cliff_oversample.npy', best_oof)
np.save(DATA_PROCESSED / 'te_lgbm_cliff_oversample.npy', te_preds)
print(f'Saved oof_lgbm_cliff_oversample.npy  ({best_label}, OOF RAE = {rae(y_tr, best_oof):.4f})')

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '59_lgbm_cliff_oversample.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print('\n== Summary ==')
print(f'  Active compounds (20x + SMILES aug): {n_active}')
print(f'  Cliff members (15x):                  {n_cliff}')
print(f'  Augmented training set size:          {len(X_aug_all):,}')
print(f'  OOF RAE (baseline):                   {rae_base_oof:.4f}')
print(f'  OOF RAE (oversampled):                {rae_over_oof:.4f}')
print(f'  Test pred std:                        {te_preds.std():.4f}')
sub['pEC50'].describe().round(3)

Training final model on 9,143 augmented samples...


Saved oof_lgbm_cliff_oversample.npy  (baseline, OOF RAE = 0.5600)
Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\59_lgbm_cliff_oversample.csv

== Summary ==
  Active compounds (20x + SMILES aug): 67
  Cliff members (15x):                  248
  Augmented training set size:          9,143
  OOF RAE (baseline):                   0.5600
  OOF RAE (oversampled):                0.5757
  Test pred std:                        0.6821


count    513.000
mean       4.820
std        0.683
min        2.365
25%        4.420
50%        4.954
75%        5.328
max        6.041
Name: pEC50, dtype: float64